<a href="https://colab.research.google.com/github/stylekkm051049-bit/Project/blob/%E0%B9%80%E0%B8%81%E0%B8%95%E0%B8%B8%E0%B8%81%E0%B8%A1%E0%B8%A5-%E0%B9%80%E0%B8%AB%E0%B8%A5%E0%B9%88%E0%B8%B2%E0%B9%80%E0%B8%AA%E0%B8%99/%E0%B8%AA%E0%B9%88%E0%B8%A7%E0%B8%99%E0%B8%97%E0%B8%B5%E0%B9%88_6_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## ขั้นที่ 6 — แปลง Object เป็น DataFrame

In [ ]:
import pandas as pd

# สร้าง dictionary สำหรับค้นหาข้อมูลการชำระเงิน
payment_dict = { p.enrollment.enrollment_id: p for p in payments}

registration_data = []

for e in enrollments:

    # หาข้อมูลการชำระเงินของรายการนี้
    payment = payment_dict.get(e.enrollment_id)

    registration_data.append({

        # ---------------- Student ----------------
        "รหัสนักเรียน": e.student.student_id,
        "ชื่อ-นามสกุล": e.student.name,
        "เพศ": e.student.gender,
        "อายุ": e.student.age,
        "ระดับชั้น": e.student.grade,
        "โรงเรียน": e.student.school,
        "เบอร์โทรติดต่อ": e.student.phone,

        # ---------------- Course ----------------
        "รหัสคอร์ส": e.course.course_id,
        "ชื่อคอร์ส": e.course.course_name,
        "รายวิชา": e.course.subject,
        "อาจารย์ผู้สอน": e.course.teacher,
        "ราคา(บาท)": e.course.price,
        "จำนวนรับสูงสุด": e.course.capacity,
        "จำนวนผู้ลงทะเบียน": e.course.enrolled_count,
        "ที่นั่งคงเหลือ": e.course.available_seats(),

        # ---------------- Enrollment ----------------
        "รหัสการลงทะเบียน": e.enrollment_id,
        "วันที่ลงทะเบียน": e.enroll_date,
        "ส่วนลด": e.discount_rate,

        # ใช้ total_tuition ตามที่คุณใช้ใน Demo
        "ยอดชำระสุทธิ (บาท)": e.total_tuition,

        "สถานะการลงทะเบียน": e.status,

        # ---------------- Payment ----------------
        "รหัสการชำระเงิน":
            payment.payment_id if payment else None,

        "จำนวนเงินที่ชำระ(บาท)":
            payment.amount if payment else None,

        "ช่องทางการชำระเงิน":
            payment.method if payment else None,

        "วันที่ชำระเงิน":
            payment.payment_date if payment else None,

        "สถานะการชำระเงิน":
            payment.status if payment else None })


# สร้าง DataFrame
registration_df = pd.DataFrame(registration_data)

# เริ่มเลขแถวที่ 1
registration_df.index = range( 1, len(registration_df) + 1)

# แสดงข้อมูล 450 รายการ
display(registration_df.head(450))

# ตรวจสอบจำนวน
print("จำนวนรายการทั้งหมด:", len(registration_df))

In [ ]:

registration_df.to_csv("ข้อมูลการลงทะเบียนคอร์สเรียน.csv",index=False,encoding="utf-8-sig")
print("บันทึก CSV สำเร็จ 450 รายการ")


In [ ]:
# โหลดข้อมูลจาก CSV ที่เพิ่งสร้าง
df_csv = pd.read_csv("ข้อมูลการลงทะเบียนคอร์สเรียน.csv")
print("โหลดข้อมูลจาก CSV สำเร็จ")
print("จำนวนข้อมูลทั้งหมด:", len(df_csv), "รายการ")
display(df_csv.head())

In [ ]:
# ตรวจสอบข้อมูลเบื้องต้น
print("ข้อมูลและชนิดของแต่ละคอลัมน์")
df_csv.info()

In [ ]:
# ดูสถิติเบื้องต้นของข้อมูล
print("สถิติเบื้องต้น")
display(df_csv.describe())

In [ ]:
# ตรวจสอบค่าว่างในแต่ละคอลัมน์
print("จำนวนค่าว่างในแต่ละคอลัมน์")
display(df_csv.isnull().sum())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.makedirs( "/content/drive/MyDrive/BAA-moire", exist_ok=True )

## ขั้นที่ 7 — สร้างฐานข้อมูล SQLite จากข้อมูลเดียวกัน

In [ ]:
import sqlite3
# เชื่อมต่อ SQLite
conn = sqlite3.connect("tutoring_school.db")
# นำ DataFrame เดียวเข้า SQLite
registration_df.to_sql( "registration", conn, if_exists="replace", index=False )

print("สร้าง SQLite Database สำเร็จ")

In [ ]:
# ทดสอบข้อมูล
check = pd.read_sql_query( "SELECT * FROM registration LIMIT 5", conn)
display(check)

In [ ]:
# ตรวจสอบว่ามีครบ 450 รายการ
check_count = pd.read_sql_query( "SELECT COUNT(*) AS จำนวนรายการ FROM registration", conn)
display(check_count)